In [8]:
training_sentences += [
    "read the book",
    "write a letter",
    "eat an apple",
    "the book is good",
    "this book is interesting",
    "a boy reads the book",
    "she writes a letter",
    "they eat food",
    "book a ticket",
    "book a cab",
    "reserve a seat"
]

training_tags += [
    ["VERB","DET","NOUN"],
    ["VERB","DET","NOUN"],
    ["VERB","DET","NOUN"],
    ["DET","NOUN","VERB","ADJ"],
    ["DET","NOUN","VERB","ADJ"],
    ["DET","NOUN","VERB","DET","NOUN"],
    ["PRON","VERB","DET","NOUN"],
    ["PRON","VERB","NOUN"],
    ["VERB","DET","NOUN"],
    ["VERB","DET","NOUN"],
    ["VERB","DET","NOUN"]
]

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Tokenize words
word_tokenizer = Tokenizer(oov_token="<OOV>")
word_tokenizer.fit_on_texts(training_sentences)
word_index = word_tokenizer.word_index

X = word_tokenizer.texts_to_sequences(training_sentences)
X = pad_sequences(X, padding='post')

# Tokenize tags manually
tag_set = set(tag for seq in training_tags for tag in seq)
tag2idx = {tag:i for i, tag in enumerate(tag_set)}
idx2tag = {i:tag for tag,i in tag2idx.items()}

y = [[tag2idx[tag] for tag in seq] for seq in training_tags]
y = pad_sequences(y, padding='post')

# Convert to categorical
from tensorflow.keras.utils import to_categorical
import numpy as np

y = np.array([to_categorical(seq, num_classes=len(tag2idx)) for seq in y])

In [10]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, TimeDistributed

model = Sequential()

model.add(Embedding(
    input_dim=len(word_index)+1,
    output_dim=8,
    input_length=X.shape[1]
))

model.add(LSTM(16, return_sequences=True))

model.add(TimeDistributed(Dense(len(tag2idx), activation='softmax')))

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ ?                      │   0 (unbuilt) │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
history = model.fit(
    X, y,
    epochs=100,
    verbose=1,
    shuffle=True
)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - accuracy: 0.4020 - loss: 1.9430
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.5098 - loss: 1.9388
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.5294 - loss: 1.9346
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5294 - loss: 1.9302
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.5098 - loss: 1.9257
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - accuracy: 0.5294 - loss: 1.9210
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5294 - loss: 1.9163
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.5294 - loss: 1.9113
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5294 - loss: 1.9062
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.5294 - loss: 1.9009
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.5294 - loss: 1.8955
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - accuracy: 0.5294 - loss

In [12]:
def predict_sentence(sentence):
    seq = word_tokenizer.texts_to_sequences([sentence])
    seq = pad_sequences(seq, maxlen=X.shape[1], padding='post')

    pred = model.predict(seq)

    pred_tags = np.argmax(pred, axis=-1)[0]

    words = sentence.split()

    result = []
    for i, word in enumerate(words):
        tag = idx2tag[pred_tags[i]]
        result.append((word, tag))

    return result

In [13]:
print(predict_sentence("read a book"))
print(predict_sentence("book a ticket"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 355ms/step
[('read', 'VERB'), ('a', 'VERB'), ('book', 'VERB')]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
[('book', 'VERB'), ('a', 'VERB'), ('ticket', 'VERB')]


Add more data

increase epochs

Increase embedding + hidden size slightly

Embedding(..., output_dim=16)

LSTM(32, return_sequences=True)
